# 03 — SCD Type 3: Previous Value

Schema:

```text
current value  -> overwritten
previous value -> old current value
history depth  -> limited to previous value
```

In [1]:
from pyspark.sql import SparkSession, functions as F
from pyspark.sql.types import *

spark = SparkSession.getActiveSession()
if spark is not None:
    spark.stop()

spark = (
    SparkSession.builder
    .appName("scd_type_3_previous_value")
    .master("spark://spark-master:7077")
    .getOrCreate()
)

spark.sparkContext.setLogLevel("WARN")
spark.version

'4.0.2'

In [2]:
current_schema = StructType([
    StructField("user_id", StringType(), False),
    StructField("segment", StringType(), False),
    StructField("previous_segment", StringType(), True),
])

incoming_schema = StructType([
    StructField("user_id", StringType(), False),
    StructField("segment", StringType(), False),
])

current = spark.createDataFrame(
    [("u1", "retail", None), ("u2", "retail", None)],
    current_schema,
)

incoming = spark.createDataFrame(
    [("u1", "premium"), ("u2", "retail"), ("u3", "retail")],
    incoming_schema,
)

current.show(truncate=False)
incoming.show(truncate=False)

+-------+-------+----------------+
|user_id|segment|previous_segment|
+-------+-------+----------------+
|u1     |retail |NULL            |
|u2     |retail |NULL            |
+-------+-------+----------------+

+-------+-------+
|user_id|segment|
+-------+-------+
|u1     |premium|
|u2     |retail |
|u3     |retail |
+-------+-------+



In [3]:
updated_existing = (
    current.alias("c")
    .join(incoming.alias("i"), on="user_id", how="left")
    .select(
        "user_id",
        F.coalesce(F.col("i.segment"), F.col("c.segment")).alias("segment"),
        F.when(
            F.col("i.segment").isNotNull() & (F.col("i.segment") != F.col("c.segment")),
            F.col("c.segment")
        ).otherwise(F.col("c.previous_segment")).alias("previous_segment"),
    )
)

new_rows = (
    incoming
    .join(current.select("user_id"), on="user_id", how="left_anti")
    .select("user_id", "segment", F.lit(None).cast(StringType()).alias("previous_segment"))
)

scd3_result = updated_existing.unionByName(new_rows).orderBy("user_id")
scd3_result.show(truncate=False)

+-------+-------+----------------+
|user_id|segment|previous_segment|
+-------+-------+----------------+
|u1     |premium|retail          |
|u2     |retail |NULL            |
|u3     |retail |NULL            |
+-------+-------+----------------+



In [4]:
totals = spark.createDataFrame(
    [
        ("current_rows", current.count()),
        ("incoming_rows", incoming.count()),
        ("new_rows", new_rows.count()),
        ("final_rows", scd3_result.count()),
    ],
    StructType([StructField("metric", StringType(), False), StructField("value", LongType(), False)])
)
totals.show(truncate=False)

+-------------+-----+
|metric       |value|
+-------------+-----+
|current_rows |2    |
|incoming_rows|3    |
|new_rows     |1    |
|final_rows   |3    |
+-------------+-----+



In [5]:
spark.stop()